In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker

from ravelry_scraper.constants import DATA_DIR


In [ ]:
con = duckdb.connect()

parquet_files = list(DATA_DIR.glob("**/*.parquet"))

# Register Parquet datasets as SQL views
for file in parquet_files:
    name = f"{file.parent.name}_{file.stem}"
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{file}')
    """)


In [ ]:
# Store list of all DuckDB views
duckdb_views = [row[0] for row in con.execute("""
    SELECT view_name
    FROM duckdb_views
""").fetchall()]

# Print all DuckDB views to verify the list
print(duckdb_views)

# Patterns

Ravelry's API exposes an endpoint for `GET /patterns/{id}.json`, which returns a Pattern record. Each Pattern describes a design and includes details like yarn requirements, price, and community ratings.


| **Notable Fields** | **Use Cases** |
| --- | --- |
| `id`, `name`, `pattern_author` | Identify Patterns and join them to Designers |
| `price`, `currency`, `free` | Compare free vs. paid patterns; account for conversions when comparing prices |
| `difficulty_average`, `difficulty_count`, `rating_average`, `rating_count` | Evaluate user perceptions (accounting for how many users contributed) |
| `projects_count`, `queued_projects_count`, `favorites_count` | How often do users start, queue, or favorite a pattern? |
| `published` | Account for how long Patterns have had to accumulate metrics |

### Challenges and Limitations

Although Patterns have a sequential primary key (`ID`), there is a gap of nearly 6m IDs between Pattern 1,370,996 and Pattern 7,281,828. However, these records have creation timestamps of "2023-10-10 15:04:38 -0400" and "2023/10/10 15:21:14 -0400" and, respectively, which is a 16 minute and 36 second gap. Given the average Pattern creation rate of ####, we do not believe our dataset to be missing any Patterns.

The most likely explanation for this jump is database maintenance or migration.

### Documentation
- https://www.ravelry.com/api#patterns_show
- https://www.ravelry.com/api#Pattern_full_result

In [ ]:
con.sql("DESCRIBE raw_patterns").df()

TODO - describe findings

In [ ]:
con.sql("""
    DESCRIBE clean_patterns
""").df()

TODO - describe findings

In [ ]:
pattern_summary_df = con.sql("""
    SELECT
        COUNT(*) AS total_patterns,
        COUNT(yardage) AS has_yardage,
        COUNT(price) AS has_price,
        ROUND(AVG(price_usd), 2) AS avg_price,
        MAX(price_usd) AS max_price,
        ROUND(AVG(rating_average), 2) AS avg_rating,
        ROUND(AVG(difficulty_average), 2) AS avg_difficulty,
        ROUND(AVG(projects_count), 2) AS avg_projects_count,
        ROUND(AVG(favorites_count), 2) AS avg_favorites_count,
        ROUND(AVG(photos_count), 2) AS avg_photos_count,
        COUNT(projects_count) AS has_projects,
        COUNT(favorites_count) AS has_favorites,
    FROM clean_patterns
""").df()

display(pattern_summary_df.select_dtypes(include="int").T)
display(pattern_summary_df.select_dtypes(include="float").T)

con.sql("""
    WITH less_than_max_price AS (
        SELECT
            ID,
            price_usd,
            price,
            currency,
        FROM clean_patterns
        WHERE price_usd < (SELECT MAX(price_usd) FROM clean_patterns)
    )
    SELECT
        ID,
        price_usd,
        price,
        currency,
    FROM less_than_max_price
    WHERE price_usd = (SELECT MAX(price_usd) FROM less_than_max_price)
""").df()

TODO - describe findings

In [ ]:
con.execute("""
    SELECT
        free,
        COUNT(*) AS total_patterns,
        COUNT(price) AS has_price,
        AVG(price) AS avg_price,
        AVG(rating_average) AS avg_rating,
        AVG(projects_count) AS avg_projects
    FROM raw_patterns
    GROUP BY free
    ORDER BY free
""").fetchdf()

In [ ]:
df = con.sql("""
    SELECT
        year(created_at) AS year,
        COUNT(id) AS count,
        SUM(COUNT(id)) OVER (
            ORDER BY year(created_at)
        )::INTEGER AS running_total
    FROM clean_patterns
    GROUP BY year
    ORDER BY year
""").df()

plt.figure(figsize=(8,6))
ax = sns.lineplot(data=df, x="year", y="count")

plt.title("Patterns Created by Year")
plt.xlabel("Year")
plt.ylabel("Count")

ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1000:g}k"))

plt.show()

display(df)

# TODO: No patterns created in 2024 or 2025?

In [ ]:
con.sql("""
    SELECT
        MAX(created_at),
        MAX(id),
    FROM clean_patterns
""").df()

In [ ]:
con.execute("""
    DESCRIBE clean_patterns
""").fetchdf()

# Yarns

In [ ]:
con.execute("""
    DESCRIBE raw_yarns
""").fetchdf()

# Extracted Numbers

TODO - describe

These record types were extracted from the core pipelines processes:
- crafts
- languages
- needle_sizes
- pattern_attributes
- pattern_categories
- pattern_types
- users
- yarn_companies
- yarn_fibers
- yarn_weights

In [ ]:
extracted_views = [view for view in duckdb_views if view.startswith("extracted")]

for view in extracted_views:
    print(f"Describing {view}:")
    display(con.execute(f"""
        SELECT column_name, column_type
        FROM (DESCRIBE {view})
    """).fetchdf())


# HW1 Example: Joining Data & getting summary stats